# Boosted-tree regression for $N_{\mathrm{part}}$ from UrQMD

This notebook trains a histogram-based gradient-boosted tree on the processed UrQMD event features. It is intentionally capped at **10,000 total events** (8,000 train, 1,000 validation, and 1,000 test) so that it can be used as a quick validation run rather than a full training campaign.

The generator impact parameter is inspected below but is **not** used as a model feature. Including it would leak collision-geometry information that is not directly available from measured final-state particles.

## 1. Imports and reproducibility

This notebook uses scikit-learn's `HistGradientBoostingRegressor`. If scikit-learn is not installed in the active kernel, install it once with `%pip install scikit-learn`, then restart the kernel.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
SEED = 42
MAX_EVENTS = 10_000
N_TRAIN = 8_000
N_VAL = 1_000
N_TEST = 1_000

assert N_TRAIN + N_VAL + N_TEST <= MAX_EVENTS
rng = np.random.default_rng(SEED)

## 2. Load the processed UrQMD table

The path search works when the notebook is launched from either the repository root or the `notebooks/` directory.

In [ ]:
relative_data_path = Path("Data/processed/urqmd_100k_tabular.npz")
search_roots = (Path.cwd(), *Path.cwd().parents)
data_path = next(
    (root / relative_data_path for root in search_roots if (root / relative_data_path).is_file()),
    None,
)

if data_path is None:
    raise FileNotFoundError(
        f"Could not find {relative_data_path}. Run this notebook from within the project tree."
    )

with np.load(data_path, allow_pickle=False) as data:
    X = data["X"].copy()  # Raw features; boosted trees do not require standardization.
    y = data["y"].copy()
    feature_names = data["feature_names"].astype(str)
    event_id = data["event_id"].copy()
    impact_parameter = data["impact_parameter"].copy()
    train_idx_full = data["train_idx"].copy()
    val_idx_full = data["val_idx"].copy()
    test_idx_full = data["test_idx"].copy()

print(f"Loaded {len(y):,} events from {data_path}")
print(f"Feature matrix: {X.shape}")
print("Features:", ", ".join(feature_names))

## 3. Confirm the impact parameter

Yes—the UrQMD data has impact parameter saved. The raw ROOT tree uses branch `b`; the processed NPZ archive uses `impact_parameter`. The checks below confirm its coverage and range.

In [ ]:
n_finite_b = np.isfinite(impact_parameter).sum()
print("impact_parameter present: yes")
print(f"Finite values: {n_finite_b:,} / {len(impact_parameter):,}")
print(
    "Range: "
    f"{np.nanmin(impact_parameter):.4f} to {np.nanmax(impact_parameter):.4f} fm"
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(impact_parameter, bins=50, histtype="step", linewidth=1.5)
ax.set(xlabel="Impact parameter b [fm]", ylabel="Events", title="UrQMD impact parameter")
plt.show()

## 4. Build a capped train/validation/test sample

Sampling is performed independently inside the preprocessing pipeline's predefined splits. This preserves split isolation while ensuring that no more than 10,000 unique events are touched by the model workflow.

In [ ]:
def capped_sample(indices, n_events, rng):
    """Sample unique row indices without exceeding the requested cap."""
    if len(indices) < n_events:
        raise ValueError(f"Requested {n_events:,} events from a split containing {len(indices):,}.")
    return rng.choice(indices, size=n_events, replace=False)


train_idx = capped_sample(train_idx_full, N_TRAIN, rng)
val_idx = capped_sample(val_idx_full, N_VAL, rng)
test_idx = capped_sample(test_idx_full, N_TEST, rng)

selected_idx = np.concatenate([train_idx, val_idx, test_idx])
assert len(selected_idx) <= MAX_EVENTS
assert len(np.unique(selected_idx)) == len(selected_idx)

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

print(f"Train:      {len(train_idx):,}")
print(f"Validation: {len(val_idx):,}")
print(f"Test:       {len(test_idx):,}")
print(f"Total:      {len(selected_idx):,} (cap: {MAX_EVENTS:,})")

## 5. Train the boosted tree

`HistGradientBoostingRegressor` uses ensembles of shallow regression trees. Early stopping reserves part of the 8,000-event training subset internally; the separate 1,000-event validation split remains untouched for model assessment.

In [ ]:
model = HistGradientBoostingRegressor(
    loss="squared_error",
    learning_rate=0.05,
    max_iter=300,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    l2_regularization=1.0,
    early_stopping=True,
    validation_fraction=0.10,
    n_iter_no_change=20,
    random_state=SEED,
)

model.fit(X_train, y_train)
print(f"Boosting iterations used: {model.n_iter_}")

## 6. Validate against a simple multiplicity baseline

The baseline is a linear fit using only `n_tracks`. It is fit on the same capped training events, so the comparison does not use any additional data.

In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
    }


val_pred = model.predict(X_val)
test_pred = model.predict(X_test)

n_tracks_col = int(np.flatnonzero(feature_names == "n_tracks")[0])
baseline_coeff = np.polyfit(X_train[:, n_tracks_col], y_train, deg=1)
val_baseline_pred = np.polyval(baseline_coeff, X_val[:, n_tracks_col])

for label, metrics in {
    "Validation boosted tree": regression_metrics(y_val, val_pred),
    "Validation n_tracks baseline": regression_metrics(y_val, val_baseline_pred),
    "Test boosted tree": regression_metrics(y_test, test_pred),
}.items():
    print(label)
    print("  " + ", ".join(f"{name}={value:.4f}" for name, value in metrics.items()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hexbin(y_val, val_pred, gridsize=35, mincnt=1, cmap="viridis")
limits = [min(y_val.min(), val_pred.min()), max(y_val.max(), val_pred.max())]
axes[0].plot(limits, limits, "r--", linewidth=1, label="Ideal")
axes[0].set(
    xlabel=r"True $N_{\mathrm{part}}$",
    ylabel=r"Predicted $N_{\mathrm{part}}$",
    title="Validation predictions",
)
axes[0].legend()

residuals = val_pred - y_val
axes[1].scatter(val_pred, residuals, s=8, alpha=0.35)
axes[1].axhline(0, color="r", linestyle="--", linewidth=1)
axes[1].set(
    xlabel=r"Predicted $N_{\mathrm{part}}$",
    ylabel="Prediction − truth",
    title="Validation residuals",
)

fig.tight_layout()
plt.show()

## 7. Permutation feature importance

Permutation importance is evaluated only on the 1,000-event validation subset. Higher values indicate a larger increase in validation error when that feature is shuffled.

In [ ]:
importance = permutation_importance(
    model,
    X_val,
    y_val,
    scoring="neg_mean_squared_error",
    n_repeats=10,
    random_state=SEED,
    n_jobs=-1,
)
order = np.argsort(importance.importances_mean)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(
    feature_names[order],
    importance.importances_mean[order],
    xerr=importance.importances_std[order],
)
ax.set(xlabel="Increase in MSE after permutation", title="Validation feature importance")
fig.tight_layout()
plt.show()

## Notes for a later full study

- Keep `impact_parameter` out of the production feature set unless the goal is explicitly a generator-level upper-bound study.
- Tune hyperparameters with cross-validation or a dedicated validation set before reporting final test performance.
- Study performance versus true $N_{\mathrm{part}}$, impact parameter, and multiplicity to identify centrality-dependent bias.
- Retrain on the full training split only after this 10,000-event validation workflow is satisfactory.